<a href="https://colab.research.google.com/github/999mashiro-pixel/nbcls/blob/main/nbcls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. 檢查 Google 分配給你的 GPU 型號
!nvidia-smi

# 2. 直接安裝 ultralytics
!pip install ultralytics

# 3. 驗證安裝
import ultralytics
ultralytics.checks()

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.4/112.6 GB disk)


In [4]:
!pip install ultralytics roboflow

In [5]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.4/112.6 GB disk)


In [6]:
import ultralytics
from ultralytics import YOLO

# 這行會印出你的硬體資訊，確保看到 "CUDA:0 (Tesla T4...)"
ultralytics.checks()

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.4/112.6 GB disk)


In [7]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="c8kjPzZ0bAGHaNJDyyrf")
project = rf.workspace("data-talent").project("hard-hat-cpajs")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Hard-hat-1 in yolov8:: 100%|██████████| 11389/11389 [00:02<00:00, 5560.84it/s] 


In [8]:
from ultralytics import YOLO

# 1. 載入預訓練模型
model = YOLO('yolov8n.pt')

# 2. 開始訓練模型！
# 請注意：data 欄位要填入下載下來的資料夾中的 data.yaml 路徑
results = model.train(
    data='/content/Hard-hat-1/data.yaml',  # 如果資料夾名稱不同，請依左側顯示的名稱修改
    epochs=25,                                   # 初步先跑 25 輪測試
    imgsz=640,                                   # 圖片輸入大小
    device=0                                     # 指定用 GPU 跑
)

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Hard-hat-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

In [13]:
# 用終極暴力指令，強制從影片中每 300 幀（約 10 秒）抽出一張圖片，存到 /content/ 目录下
!ffmpeg -i /content/ML_final_data.mp4 -vf "select=not(mod(n\,300))" -vsync vfr /content/video_frame_%03d.jpg

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [14]:
# 讓模型去辨識剛剛抽出來的影片圖片
import glob

extracted_frames = glob.glob('/content/video_frame_*.jpg')

for i, img_path in enumerate(extracted_frames[:3]): # 測前 3 張
    res = model_best.predict(source=img_path, conf=0.25, device=0)
    plot_img = res[0].plot()

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(plot_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Video Frame Test {i+1}")
    plt.axis('off')
    plt.show()